## Importing Libraries

In [9]:
import sys
from pathlib import Path
from timescaledb.hyperfunctions import time_bucket
from pprint import pprint
from sqlalchemy import func

#### Resolving Path

In [3]:
src_path = Path('..').resolve()
sys.path.insert(0, str(src_path))

print(sys.path[0])  # confirm

C:\Users\adans\Documents\Projetos-pessoais\analytics-api


In [4]:
from src.api.events.models import EventModel
from src.api.db.session import engine
from sqlmodel import Session, select

## Query 10 oldest events

In [8]:
with Session(engine) as session:
    query = select(EventModel).order_by(EventModel.updated_at.asc()).limit(10)
    results = session.exec(query).all()
    pprint(results)

2026-03-03 14:34:06,701 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-03 14:34:06,704 INFO sqlalchemy.engine.Engine SELECT eventmodel.id, eventmodel.time, eventmodel.page, eventmodel.description, eventmodel.updated_at 
FROM eventmodel ORDER BY eventmodel.updated_at ASC 
 LIMIT %(param_1)s
2026-03-03 14:34:06,705 INFO sqlalchemy.engine.Engine [cached since 2.87s ago] {'param_1': 10}
[EventModel(description=None, id=1, page='/pricing', updated_at=datetime.datetime(2026, 3, 3, 14, 16, 55, 644006, tzinfo=datetime.timezone.utc), time=datetime.datetime(2026, 3, 3, 14, 16, 55, 643238, tzinfo=datetime.timezone.utc)),
 EventModel(description=None, id=2, page='/about', updated_at=datetime.datetime(2026, 3, 3, 14, 17, 23, 863804, tzinfo=datetime.timezone.utc), time=datetime.datetime(2026, 3, 3, 14, 17, 23, 863745, tzinfo=datetime.timezone.utc)),
 EventModel(description=None, id=3, page='/contact', updated_at=datetime.datetime(2026, 3, 3, 14, 17, 24, 651498, tzinfo=datetime.timezone.utc),

In [5]:
compiled_query = query.compile(compile_kwargs={"literal_binds": True})

print(compiled_query)

SELECT eventmodel.id, eventmodel.time, eventmodel.page, eventmodel.description, eventmodel.updated_at 
FROM eventmodel ORDER BY eventmodel.updated_at ASC
 LIMIT 10


## Query with time buckets

In [11]:
with Session(engine) as session:
    bucket = time_bucket("1 day", EventModel.time)

    query = (
        select(
            bucket,
            EventModel.page,
        )
        .order_by(EventModel.updated_at.asc())
    )

    compiled_query = query.compile(compile_kwargs={"literal_binds": True})
    results = session.exec(query).fetchall()
    pprint(results)

2026-03-03 14:37:33,390 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-03 14:37:33,398 INFO sqlalchemy.engine.Engine SELECT time_bucket('1 day'::interval, eventmodel.time) AS time_bucket_1, eventmodel.page 
FROM eventmodel ORDER BY eventmodel.updated_at ASC
2026-03-03 14:37:33,427 INFO sqlalchemy.engine.Engine [cached since 243.5s ago] {}


[(datetime.datetime(2026, 3, 3, 0, 0, tzinfo=datetime.timezone.utc), '/pricing'),
 (datetime.datetime(2026, 3, 3, 0, 0, tzinfo=datetime.timezone.utc), '/about'),
 (datetime.datetime(2026, 3, 3, 0, 0, tzinfo=datetime.timezone.utc), '/contact'),
 (datetime.datetime(2026, 3, 3, 0, 0, tzinfo=datetime.timezone.utc), '/pages'),
 (datetime.datetime(2026, 3, 3, 0, 0, tzinfo=datetime.timezone.utc), '/about'),
 (datetime.datetime(2026, 3, 3, 0, 0, tzinfo=datetime.timezone.utc), '/pages'),
 (datetime.datetime(2026, 3, 3, 0, 0, tzinfo=datetime.timezone.utc), '/pricing'),
 (datetime.datetime(2026, 3, 3, 0, 0, tzinfo=datetime.timezone.utc), '/pages'),
 (datetime.datetime(2026, 3, 3, 0, 0, tzinfo=datetime.timezone.utc), '/contact'),
 (datetime.datetime(2026, 3, 3, 0, 0, tzinfo=datetime.timezone.utc), '/pricing'),
 (datetime.datetime(2026, 3, 3, 0, 0, tzinfo=datetime.timezone.utc), '/contact'),
 (datetime.datetime(2026, 3, 3, 0, 0, tzinfo=datetime.timezone.utc), '/pages'),
 (datetime.datetime(2026, 3,

## Query with time buckets and Count

In [12]:
with Session(engine) as session:
    bucket = time_bucket("1 day", EventModel.time)

    query = (
        select(
            bucket,
            EventModel.page,
            func.count()
        )
        .group_by(
            bucket,
            EventModel.page,)
    )

    compiled_query = query.compile(compile_kwargs={"literal_binds": True})
    results = session.exec(query).fetchall()
    pprint(results)

2026-03-03 14:39:21,030 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-03 14:39:21,048 INFO sqlalchemy.engine.Engine SELECT time_bucket('1 day'::interval, eventmodel.time) AS time_bucket_1, eventmodel.page, count(*) AS count_1 
FROM eventmodel GROUP BY time_bucket('1 day'::interval, eventmodel.time), eventmodel.page
2026-03-03 14:39:21,050 INFO sqlalchemy.engine.Engine [generated in 0.00316s] {}
[(datetime.datetime(2026, 3, 3, 0, 0, tzinfo=datetime.timezone.utc), '/about', 471),
 (datetime.datetime(2026, 3, 3, 0, 0, tzinfo=datetime.timezone.utc), '/pages', 419),
 (datetime.datetime(2026, 3, 3, 0, 0, tzinfo=datetime.timezone.utc), '/contact', 472),
 (datetime.datetime(2026, 3, 3, 0, 0, tzinfo=datetime.timezone.utc), '/pricing', 424)]
2026-03-03 14:39:21,151 INFO sqlalchemy.engine.Engine ROLLBACK


## Query with time buckets, count and selecting pages

In [13]:
with Session(engine) as session:
    bucket = time_bucket("1 day", EventModel.time)
    pages = ['/about', '/contact']

    query = (
        select(
            bucket,
            EventModel.page,
            func.count()
        )
        .where(
            EventModel.page.in_(pages)
        )
        .group_by(
            bucket,
            EventModel.page,)
    )

    compiled_query = query.compile(compile_kwargs={"literal_binds": True})
    results = session.exec(query).fetchall()
    pprint(results)

2026-03-03 14:48:19,676 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-03 14:48:19,696 INFO sqlalchemy.engine.Engine SELECT time_bucket('1 day'::interval, eventmodel.time) AS time_bucket_1, eventmodel.page, count(*) AS count_1 
FROM eventmodel 
WHERE eventmodel.page IN (%(page_1_1)s, %(page_1_2)s) GROUP BY time_bucket('1 day'::interval, eventmodel.time), eventmodel.page
2026-03-03 14:48:19,700 INFO sqlalchemy.engine.Engine [generated in 0.00639s] {'page_1_1': '/about', 'page_1_2': '/contact'}
[(datetime.datetime(2026, 3, 3, 0, 0, tzinfo=datetime.timezone.utc), '/about', 471),
 (datetime.datetime(2026, 3, 3, 0, 0, tzinfo=datetime.timezone.utc), '/contact', 472)]
2026-03-03 14:48:19,989 INFO sqlalchemy.engine.Engine ROLLBACK


## Query with time buckets, count, selecting pages and timedelta

In [ ]:
from datetime import datetime, timedelta, timezone

with Session(engine) as session:
    bucket = time_bucket("1 day", EventModel.time)
    pages = ['/about', '/contact']
    start = datetime.now(timezone.utc) - timedelta(hours=1)
    finish = datetime.now(timezone.utc) + timedelta(hours=1)

    query = (
        select(
            bucket,
            EventModel.page,
            func.count()
        )
        .where(
            EventModel.time > start,
            EventModel.time <= finish,
            EventModel.page.in_(pages)
        )
        .group_by(
            bucket,
            EventModel.page,)
    )

    compiled_query = query.compile(compile_kwargs={"literal_binds": True})
    results = session.exec(query).fetchall()
    pprint(results)